# Notebook 11: Medallion Architecture — Master Bronze & Silver Layer Inspection
**Course:** DAT535 - Advanced Cloud Computing and Big Data  
**Authors:** Group 13  
**Status:** Executed & Verified against Production Bronze & Silver Storage

---

## Overview & Notebook Purpose
This comprehensive master inspection notebook provides a complete, unified walkthrough of the **DAT535 Medallion Architecture** across both the **Bronze** and **Silver** layers:

1. **Whole Bronze Layer Inspection**:
   - **NYC 311 Bronze Table** (`data/bronze/nyc311/`): 91,217 raw records, partition layouts, and Tier 2 audit metadata.
   - **Weather Bronze Table** (`data/bronze/weather/`): 4,560 hourly ERA5-Land reanalysis observations across 19 native NYC grid cells.
2. **Deep-Dive into Architectural & Engineering Decisions (Live Code Blocks)**:
   - **Decision 1: Timezone Localization (`America/New_York` $\rightarrow$ `UTC`)** — dynamic DST handling vs. naive parsing.
   - **Decision 2: Change Data Capture (CDC)** — sliding lookback window (`--lookback-days 14`) & Silver deduplication/upsert.
   - **Decision 3: Cumulative & Lagged Weather Features** — rolling 3h, 6h, 24h precipitation, peak gusts, and temperature extremes.
   - **Decision 4: Open Ticket Semantics & Queue Age** — `is_ticket_closed`, `ticket_status_category`, and `open_ticket_age_days`.
   - **Decision 5: Bronze Partition Compaction** — coalescing micro-batch small files with zero row loss.
   - **Decision 6: Gentle Quality Cleaning & Physical Sanity Limits** — column nullification vs. row deletion.
   - **Decision 7: 3-Level Hierarchical Spatial Imputation** — GPS $\rightarrow$ Census ZCTA Centroid $\rightarrow$ Borough Centroid.
   - **Decision 8: Quarantine Segregation** — preserving corrupted records in quarantine without pipeline crashes.
3. **Whole Silver Layer Inspection**:
   - **Clean NYC 311 Silver Table** (`data/silver/nyc311/`): 91,167 cleaned and standardized records.
   - **Clean Weather Silver Table** (`data/silver/weather/`): 4,560 hourly metric observations with rolling window features.
   - **Unified Fact Table** (`data/silver/311_weather_joined/`): 91,167 fact rows enriched via broadcast join (86.4% match rate).
4. **Summary Comparison Matrix (Bronze vs. Silver)**: Side-by-side technical specification ledger.


In [1]:
import glob
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Configure Pandas display formatting
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# Set base project paths
current_dir = Path(".").resolve()
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
elif (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
QUARANTINE_DIR = SILVER_DIR / "quarantine"
METADATA_DIR = DATA_DIR / "metadata"
REFERENCE_DIR = DATA_DIR / "reference"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Bronze Directory: {BRONZE_DIR} (Exists: {BRONZE_DIR.exists()})")
print(f"Silver Directory: {SILVER_DIR} (Exists: {SILVER_DIR.exists()})")
print(f"Quarantine Directory: {QUARANTINE_DIR} (Exists: {QUARANTINE_DIR.exists()})")


Project Root: C:\Media\UiS\DAT535
Bronze Directory: C:\Media\UiS\DAT535\data\bronze (Exists: True)
Silver Directory: C:\Media\UiS\DAT535\data\silver (Exists: True)
Quarantine Directory: C:\Media\UiS\DAT535\data\silver\quarantine (Exists: True)


## Part 1: Inspecting the Whole BRONZE Layer
The Bronze layer is an **immutable, append-only raw data repository**. In Bronze:
* Data is stored in **native Parquet** format for efficient columnar storage and snappy compression.
* Schema types match raw ingested inputs (strings/objects for timestamps and coordinates).
* **Tier 2 System Audit Metadata** columns are injected into every record:
  * `_batch_id`: Ingestion batch UUID / identifier.
  * `_ingested_at_utc`: UTC timestamp when the ingestion runner processed the batch.
  * `_source_file`: Relative path to raw landing JSON.GZ archive.
  * `_source_name`: Upstream data provider (`nyc_open_data_311` or `open_meteo_era5`).
  * `_raw_row_hash`: Cryptographic SHA-256 fingerprint computed across all raw source attributes.


In [2]:
# Load all Bronze NYC 311 Parquet files across year/month partitions
bronze_311_files = sorted(BRONZE_DIR.glob("nyc311/**/*.parquet"))
print(f"Found {len(bronze_311_files)} Bronze 311 partition file(s):")
for f in bronze_311_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  * {f.relative_to(PROJECT_ROOT)} ({size_mb:.2f} MB)")

df_bronze_311 = pd.concat([pd.read_parquet(f) for f in bronze_311_files], ignore_index=True)
print(f"\nTotal Bronze 311 Records: {len(df_bronze_311):,}")
print(f"Total Bronze 311 Columns: {len(df_bronze_311.columns)}")

# Display Sample of Raw Bronze 311 Table
key_sample_cols = [
    "unique_key", "created_date", "closed_date", "agency", 
    "complaint_type", "descriptor", "borough", "incident_zip", 
    "latitude", "longitude", "_batch_id", "_ingested_at_utc", "_raw_row_hash"
]
print("\n--- Sample: Raw Bronze 311 Table (Top 5 Records) ---")
display(df_bronze_311[key_sample_cols].head(5))

print("\n--- Bronze 311 Data Types & Unparsed String State ---")
print(df_bronze_311[["created_date", "closed_date", "latitude", "longitude", "incident_zip"]].dtypes)


Found 2 Bronze 311 partition file(s):
  * data\bronze\nyc311\year=2026\month=07\part_311_20261023_004_0000.parquet (1.88 MB)
  * data\bronze\nyc311\year=2026\month=10\part_compacted_0000.parquet (12.47 MB)

Total Bronze 311 Records: 91,217
Total Bronze 311 Columns: 51

--- Sample: Raw Bronze 311 Table (Top 5 Records) ---


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,latitude,longitude,_batch_id,_ingested_at_utc,_raw_row_hash
0,69734449,2026-07-15T18:35:39.000,2026-07-15T21:40:32.000,NYPD,Noise - Residential,Banging/Pounding,BRONX,10456,40.834095700581216,-73.90532049838446,311_20261023_004,2026-10-23T09:33:21Z,b457eaa98ea7336fb17a93064d06bf9f76684691a603e3...
1,69734931,2026-07-15T18:35:48.000,2026-07-16T17:52:28.000,DPR,Damaged Tree,Entire Tree Has Fallen Down,BROOKLYN,11209,40.62660966664778,-74.03405790611687,311_20261023_004,2026-10-23T09:33:21Z,c38dc1302786d7feb3f8c12035827de4af98aef581947e...
2,69726765,2026-07-15T18:36:00.000,2026-07-15T18:36:00.000,DEP,Water System,Fire Hydrant Emergency (FHE),MANHATTAN,10026,40.80185697136951,-73.95723787920869,311_20261023_004,2026-10-23T09:33:21Z,b42cccd3d28fc13d164eb893e363ab0b2c0f54bc543549...
3,69727719,2026-07-15T18:36:39.000,NaN,DPR,Damaged Tree,Branch or Limb Has Fallen Down,BROOKLYN,11214,40.603609908229664,-74.01260435844628,311_20261023_004,2026-10-23T09:33:21Z,b6272a58413a0ffaf6612649b8d8e93f24f67fec264825...
4,69729114,2026-07-15T18:36:40.000,2026-07-15T21:12:48.000,NYPD,Abandoned Vehicle,With License Plate,BROOKLYN,11236,40.63838235103757,-73.91393942105312,311_20261023_004,2026-10-23T09:33:21Z,bcdaf0842aa24e211e6e682929d19ffbd30f4f57847951...



--- Bronze 311 Data Types & Unparsed String State ---
created_date    str
closed_date     str
latitude        str
longitude       str
incident_zip    str
dtype: object


In [3]:
# Load all Bronze Weather Parquet files across the 19 NYC grid cells
bronze_weather_files = sorted(BRONZE_DIR.glob("weather/**/*.parquet"))
print(f"Found {len(bronze_weather_files)} Bronze Weather partition file(s) across grid cells.")

df_bronze_weather = pd.concat([pd.read_parquet(f) for f in bronze_weather_files], ignore_index=True)
print(f"Total Bronze Weather Observations: {len(df_bronze_weather):,}")
print(f"Total Bronze Weather Columns: {len(df_bronze_weather.columns)}")
print(f"Distinct Native Grid Cells: {df_bronze_weather['grid_id'].nunique()}")

# Display Sample of Raw Bronze Weather Table
w_cols = [
    "grid_id", "time", "temperature_2m", "precipitation", 
    "wind_speed_10m", "wind_gusts_10m", "surface_pressure", 
    "_batch_id", "_ingested_at_utc", "_raw_row_hash"
]
print("\n--- Sample: Raw Bronze Weather Table (Top 5 Observations) ---")
display(df_bronze_weather[w_cols].head(5))

# Summary statistics of raw observations
print("\n--- Bronze Weather Sensor Summary Statistics ---")
display(df_bronze_weather[["temperature_2m", "precipitation", "wind_speed_10m", "wind_gusts_10m", "surface_pressure"]].describe())


Found 19 Bronze Weather partition file(s) across grid cells.
Total Bronze Weather Observations: 4,560
Total Bronze Weather Columns: 26
Distinct Native Grid Cells: 19

--- Sample: Raw Bronze Weather Table (Top 5 Observations) ---


,grid_id,time,temperature_2m,precipitation,wind_speed_10m,wind_gusts_10m,surface_pressure,_batch_id,_ingested_at_utc,_raw_row_hash
0,GRID_40.5_-74.1,2026-10-18T00:00,8.700,0.000,15.000,27.400,1027.300,wth_20261023_001,2026-10-23T09:19:29Z,386c495c428e19a601dd8c1cb03f30ef1c30df640726d8...
1,GRID_40.5_-74.1,2026-10-18T01:00,8.400,0.000,16.300,32.800,1028.200,wth_20261023_001,2026-10-23T09:19:29Z,03caed9ddf0c60f0ced539aa7badb8e8de340a2c697d8e...
2,GRID_40.5_-74.1,2026-10-18T02:00,7.900,0.000,16.700,33.800,1029.000,wth_20261023_001,2026-10-23T09:19:29Z,a1271d9dd5d48e4702429cb4ffb5be270699ddb90abb12...
3,GRID_40.5_-74.1,2026-10-18T03:00,7.400,0.000,14.800,33.500,1030.000,wth_20261023_001,2026-10-23T09:19:29Z,fd2e19dd70bea34b73c5bd77ecd22f372d396d3a5e6727...
4,GRID_40.5_-74.1,2026-10-18T04:00,6.900,0.000,12.500,29.500,1030.800,wth_20261023_001,2026-10-23T09:19:29Z,9362f203fd636ef054a3f06a99fb6afb9f088e05d43ade...



--- Bronze Weather Sensor Summary Statistics ---


,temperature_2m,precipitation,wind_speed_10m,wind_gusts_10m,surface_pressure
count,4560.000,4560.000,4560.000,4560.000,4560.000
mean,18.501,0.055,11.083,26.595,1017.251
std,7.931,0.483,4.663,10.289,8.524
min,3.000,0.000,0.200,2.200,997.000
25%,12.300,0.000,7.300,19.100,1010.600
50%,17.300,0.000,11.400,26.600,1016.200
75%,24.700,0.000,14.800,34.200,1023.100
max,37.600,15.700,23.100,53.300,1036.800


## Part 2: Engineering Decisions & Transformation Rules (Before vs. After)
This section demonstrates the 8 key architectural decisions, validation algorithms, and data cleansing logic applied between Bronze and Silver.


In [4]:
# DECISION 1: Timezone Localization (America/New_York -> UTC)
# Problem Before: Naive pd.to_datetime(utc=True) treated Eastern time as UTC, causing a 4-5 hr phase shift!
# Solution: Localize to America/New_York, then convert to UTC.

sample_dates = pd.Series([
    "2026-07-15T14:30:00.000",  # Summer: Eastern Daylight Time (EDT = UTC - 4h)
    "2026-01-15T14:30:00.000",  # Winter: Eastern Standard Time (EST = UTC - 5h)
])

# 1. Flawed Naive Approach
naive_utc = pd.to_datetime(sample_dates, utc=True)

# 2. Correct IANA Localized Approach
def to_utc_datetime(series: pd.Series, local_tz: str = "America/New_York") -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce")
    if dt.dt.tz is None:
        return dt.dt.tz_localize(local_tz, ambiguous="NaT", nonexistent="shift_forward").dt.tz_convert("UTC")
    return dt.dt.tz_convert("UTC")

correct_utc = to_utc_datetime(sample_dates)

comparison_df = pd.DataFrame({
    "Raw Input String": sample_dates,
    "Season": ["Summer (EDT)", "Winter (EST)"],
    "Naive Parsing (Before)": naive_utc,
    "Correct IANA UTC (After)": correct_utc,
    "Hourly Weather Join Key": correct_utc.dt.floor("h"),
})
print("--- Timezone Conversion: Before vs. After Comparison ---")
display(comparison_df)


--- Timezone Conversion: Before vs. After Comparison ---


,Raw Input String,Season,Naive Parsing (Before),Correct IANA UTC (After),Hourly Weather Join Key
0,2026-07-15T14:30:00.000,Summer (EDT),2026-07-15 14:30:00+00:00,2026-07-15 18:30:00+00:00,2026-07-15 18:00:00+00:00
1,2026-01-15T14:30:00.000,Winter (EST),2026-01-15 14:30:00+00:00,2026-01-15 19:30:00+00:00,2026-01-15 19:00:00+00:00


In [5]:
# DECISION 2: Change Data Capture (CDC) & Late-Arriving Ticket Updates
# Problem Before: created_date > watermark missed status updates (closures) to older tickets.
# Solution: Ingestion queries with a 14-day sliding lookback buffer, and Silver merges on unique_key.

# Simulate late closure update
mock_ingestion_stream = pd.DataFrame([
    # Initial batch: Ticket 1001 created on Oct 10 as OPEN
    {"unique_key": "1001", "created_date": "2026-10-10T10:00:00.000", "closed_date": None, "status": "Open", "_ingested_at_utc": "2026-10-10T10:05:00Z"},
    # Incremental batch 2 days later: Ticket 1001 resolved and closed!
    {"unique_key": "1001", "created_date": "2026-10-10T10:00:00.000", "closed_date": "2026-10-12T14:00:00.000", "status": "Closed", "_ingested_at_utc": "2026-10-12T14:05:00Z"},
    # New ticket in batch 2
    {"unique_key": "1002", "created_date": "2026-10-12T12:00:00.000", "closed_date": None, "status": "Open", "_ingested_at_utc": "2026-10-12T14:05:00Z"},
])

print("--- Multi-Version Bronze Records (Before Deduplication) ---")
display(mock_ingestion_stream)

# Silver Deduplication Logic: Sort by _ingested_at_utc descending, keep first
df_upserted = mock_ingestion_stream.sort_values("_ingested_at_utc", ascending=False).drop_duplicates(subset=["unique_key"], keep="first")

print("\n--- Clean Silver Records (After Deduplication & Upsert) ---")
display(df_upserted)


--- Multi-Version Bronze Records (Before Deduplication) ---


,unique_key,created_date,closed_date,status,_ingested_at_utc
0,1001,2026-10-10T10:00:00.000,NaN,Open,2026-10-10T10:05:00Z
1,1001,2026-10-10T10:00:00.000,2026-10-12T14:00:00.000,Closed,2026-10-12T14:05:00Z
2,1002,2026-10-12T12:00:00.000,NaN,Open,2026-10-12T14:05:00Z



--- Clean Silver Records (After Deduplication & Upsert) ---


,unique_key,created_date,closed_date,status,_ingested_at_utc
1,1001,2026-10-10T10:00:00.000,2026-10-12T14:00:00.000,Closed,2026-10-12T14:05:00Z
2,1002,2026-10-12T12:00:00.000,NaN,Open,2026-10-12T14:05:00Z


In [6]:
# DECISION 3: Cumulative & Lagged Weather Features
# Problem Before: Instantaneous 1-hour weather missed cumulative rainfall (floods) and prolonged cold (freezing pipes).
# Solution: Rolling window aggregations partitioned by grid_id.

# Pick one grid cell and compute rolling accumulations
sample_grid = df_bronze_weather[df_bronze_weather["grid_id"] == "GRID_40.7_-74.0"].sort_values("time").copy()
sample_grid["timestamp_utc"] = pd.to_datetime(sample_grid["time"], utc=True)

# Rolling features
sample_grid["precip_past_3h_mm"] = sample_grid["precipitation"].rolling(3, min_periods=1).sum().round(2)
sample_grid["precip_past_6h_mm"] = sample_grid["precipitation"].rolling(6, min_periods=1).sum().round(2)
sample_grid["precip_past_24h_mm"] = sample_grid["precipitation"].rolling(24, min_periods=1).sum().round(2)
sample_grid["max_wind_gust_past_6h_ms"] = sample_grid["wind_gusts_10m"].rolling(6, min_periods=1).max().round(2)
sample_grid["min_temp_past_24h_c"] = sample_grid["temperature_2m"].rolling(24, min_periods=1).min().round(2)

print("--- Cumulative Weather Features: Sample Grid Cell GRID_40.7_-74.0 ---")
display(sample_grid[["timestamp_utc", "temperature_2m", "precipitation", "precip_past_3h_mm", "precip_past_24h_mm", "wind_gusts_10m", "max_wind_gust_past_6h_ms"]].head(10))


--- Cumulative Weather Features: Sample Grid Cell GRID_40.7_-74.0 ---


,timestamp_utc,temperature_2m,precipitation,precip_past_3h_mm,precip_past_24h_mm,wind_gusts_10m,max_wind_gust_past_6h_ms
3048,2026-07-15 00:00:00+00:00,31.600,0.000,0.000,0.000,40.300,40.300
3049,2026-07-15 01:00:00+00:00,30.700,0.000,0.000,0.000,34.200,40.300
3050,2026-07-15 02:00:00+00:00,29.800,0.000,0.000,0.000,34.600,40.300
3051,2026-07-15 03:00:00+00:00,28.900,0.000,0.000,0.000,33.800,40.300
3052,2026-07-15 04:00:00+00:00,28.300,0.000,0.000,0.000,31.300,40.300
3053,2026-07-15 05:00:00+00:00,27.900,0.000,0.000,0.000,29.200,40.300
3054,2026-07-15 06:00:00+00:00,27.400,0.000,0.000,0.000,26.600,34.600
3055,2026-07-15 07:00:00+00:00,27.000,0.000,0.000,0.000,23.400,34.600
3056,2026-07-15 08:00:00+00:00,26.300,0.000,0.000,0.000,22.000,33.800
3057,2026-07-15 09:00:00+00:00,25.800,0.000,0.000,0.000,19.400,31.300


In [7]:
# DECISION 4: Open Ticket Semantics & Active Queue Age Tracking
# Problem Before: Open tickets had resolution_duration = NaN, causing analysts to discard them and bias reports.
# Solution: Add is_ticket_closed (bool), ticket_status_category ('OPEN'/'CLOSED'), and open_ticket_age_days.

df_311_eval = df_bronze_311.copy()
df_311_eval["created_at_utc"] = to_utc_datetime(df_311_eval["created_date"])
df_311_eval["closed_at_utc"] = to_utc_datetime(df_311_eval.get("closed_date"))

df_311_eval["is_ticket_closed"] = df_311_eval["closed_at_utc"].notna()
df_311_eval["ticket_status_category"] = np.where(df_311_eval["is_ticket_closed"], "CLOSED", "OPEN")

# Duration in hours (for closed tickets)
duration_secs = (df_311_eval["closed_at_utc"] - df_311_eval["created_at_utc"]).dt.total_seconds()
df_311_eval["resolution_duration_hours"] = np.where(df_311_eval["is_ticket_closed"], (duration_secs / 3600.0).round(2), np.nan)

# Age in days (for open tickets)
now_utc = pd.Timestamp.now(tz="UTC")
age_secs = (now_utc - df_311_eval["created_at_utc"]).dt.total_seconds()
df_311_eval["open_ticket_age_days"] = np.where(~df_311_eval["is_ticket_closed"], (age_secs / 86400.0).round(2), np.nan)

# Display Ticket Status Breakdown
status_breakdown = df_311_eval["ticket_status_category"].value_counts()
status_pct = df_311_eval["ticket_status_category"].value_counts(normalize=True) * 100
summary_status = pd.DataFrame({"Ticket Count": status_breakdown, "Percentage (%)": status_pct.round(2)})

print("--- Open vs. Closed Ticket Semantics Breakdown ---")
display(summary_status)

print("\n--- Sample: Closed Tickets with Resolution Duration ---")
display(df_311_eval[df_311_eval["is_ticket_closed"]][["unique_key", "created_at_utc", "closed_at_utc", "ticket_status_category", "resolution_duration_hours"]].head(3))

print("\n--- Sample: Open Tickets with Active Queue Age ---")
display(df_311_eval[~df_311_eval["is_ticket_closed"]][["unique_key", "created_at_utc", "closed_at_utc", "ticket_status_category", "open_ticket_age_days"]].head(3))


--- Open vs. Closed Ticket Semantics Breakdown ---


,Ticket Count,Percentage (%)
ticket_status_category,,
CLOSED,64711,70.940
OPEN,26506,29.060



--- Sample: Closed Tickets with Resolution Duration ---


,unique_key,created_at_utc,closed_at_utc,ticket_status_category,resolution_duration_hours
0,69734449,2026-07-15 22:35:39+00:00,2026-07-16 01:40:32+00:00,CLOSED,3.080
1,69734931,2026-07-15 22:35:48+00:00,2026-07-16 21:52:28+00:00,CLOSED,23.280
2,69726765,2026-07-15 22:36:00+00:00,2026-07-15 22:36:00+00:00,CLOSED,0.000



--- Sample: Open Tickets with Active Queue Age ---


,unique_key,created_at_utc,closed_at_utc,ticket_status_category,open_ticket_age_days
3,69727719,2026-07-15 22:36:39+00:00,NaT,OPEN,102.560
6,69727731,2026-07-15 22:36:53+00:00,NaT,OPEN,102.560
26,69726370,2026-07-15 22:38:27+00:00,NaT,OPEN,102.560


In [8]:
# DECISION 5: Bronze Partition Sizing & Compaction Utility
# Problem Before: Micro-batches write small ~20KB files, causing the Small Files Problem (high I/O latency, inode bloat).
# Solution: Compaction engine coalesces small files within leaf partitions with strict row invariance verification.

from src.utils.compaction import compact_partitions

# Run dry-run scan on bronze to inspect current compaction status
compaction_audit = compact_partitions(BRONZE_DIR, target_size_mb=64, dry_run=True)

print("--- Bronze Partition Compaction Audit ---")
print(f"Total Partitions Scanned : {compaction_audit['total_partitions_scanned']}")
print(f"Partitions Needing Merge : {compaction_audit['partitions_compacted']}")
print(f"Small Files Eliminated   : {compaction_audit['files_eliminated']}")
print(f"Total Rows Verified      : {compaction_audit['total_rows_verified']:,}")


2026-10-26 13:04:07,115 [INFO] Scanning 21 partition directories under C:\Media\UiS\DAT535\data\bronze (Target size: 64MB)...
2026-10-26 13:04:07,117 [INFO] Compaction Complete: 0 partitions compacted | 0 small files eliminated | 0 rows preserved | 0.0 KB saved | 0.01s


--- Bronze Partition Compaction Audit ---
Total Partitions Scanned : 21
Partitions Needing Merge : 0
Small Files Eliminated   : 0
Total Rows Verified      : 0


In [9]:
# DECISION 6: Gentle Quality Cleaning & Physical Limits (Column Nullification vs. Row Deletion)
# Problem Before: Dropping an entire row because of one corrupted sensor reading discards valid data.
# Solution: Nullify only the corrupt field, set has_corrupted_sensor_fields=True, and tag extreme events.

from src.silver.config import WEATHER_HARD_LIMITS, WEATHER_EXTREME_THRESHOLDS

mock_weather_dirty = pd.DataFrame([
    # Valid normal observation
    {"time": "2026-10-18T10:00", "temperature_2m": 15.0, "precipitation": 0.0, "wind_speed_10m": 5.0, "surface_pressure": 1013.0},
    # Corrupt sensor glitch: temperature = -999.0 (sentinel value)
    {"time": "2026-10-18T11:00", "temperature_2m": -999.0, "precipitation": 2.0, "wind_speed_10m": 8.0, "surface_pressure": 1012.0},
    # Real extreme weather: torrential downpour (35 mm/hr) & cold wave (-26°C)
    {"time": "2026-10-18T12:00", "temperature_2m": -26.0, "precipitation": 35.0, "wind_speed_10m": 25.0, "surface_pressure": 980.0},
])

df_w_clean = mock_weather_dirty.copy()

# 1. Nullify only corrupt fields
corrupt_t = ~df_w_clean["temperature_2m"].between(WEATHER_HARD_LIMITS["temp_min"], WEATHER_HARD_LIMITS["temp_max"])
df_w_clean.loc[corrupt_t, "temperature_2m"] = np.nan
df_w_clean["has_corrupted_sensor_fields"] = corrupt_t

# 2. Tag real extremes (PRESERVED INTACT)
cond_cold = df_w_clean["temperature_2m"] < WEATHER_EXTREME_THRESHOLDS["temp_extreme_cold"]
cond_torrential = df_w_clean["precipitation"] >= WEATHER_EXTREME_THRESHOLDS["precip_torrential_hr"]
df_w_clean["is_extreme_weather"] = (cond_cold | cond_torrential).fillna(False)

print("--- Gentle Cleaning Demonstration: Raw vs. Cleaned ---")
display(df_w_clean)


--- Gentle Cleaning Demonstration: Raw vs. Cleaned ---


,time,temperature_2m,precipitation,wind_speed_10m,surface_pressure,has_corrupted_sensor_fields,is_extreme_weather
0,2026-10-18T10:00,15.000,0.000,5.000,1013.000,False,False
1,2026-10-18T11:00,NaN,2.000,8.000,1012.000,True,False
2,2026-10-18T12:00,-26.000,35.000,25.000,980.000,False,True


In [10]:
# DECISION 7: 3-Level Hierarchical Spatial Imputation
# Level 1: Native GPS Coordinates within NYC Bounds [40.40, 41.05] x [-74.30, -73.65]
# Level 2: U.S. Census Bureau 2026 ZCTA Centroid Fallback (data/reference/nyc_zip_centroids.parquet)
# Level 3: NYC Borough Centroid Fallback

from src.silver.clean_311 import load_zip_reference
from src.silver.config import BOROUGH_CENTROIDS, NYC_LAT_MIN, NYC_LAT_MAX, NYC_LON_MIN, NYC_LON_MAX

zip_ref = load_zip_reference()
print(f"Loaded Census ZCTA Reference: {len(zip_ref)} NYC ZIP codes mapped to centroids.")
display(zip_ref.head(4))

# Inspect live distribution of spatial resolution in Silver 311
silver_311_files = sorted(SILVER_DIR.glob("nyc311/**/*.parquet"))
df_silver_311 = pd.concat([pd.read_parquet(f) for f in silver_311_files], ignore_index=True)

spatial_counts = df_silver_311["spatial_resolution_level"].value_counts()
spatial_pct = df_silver_311["spatial_resolution_level"].value_counts(normalize=True) * 100
spatial_summary = pd.DataFrame({"Record Count": spatial_counts, "Percentage (%)": spatial_pct.round(2)})

print("\n--- Silver 311 Spatial Resolution Breakdown ---")
display(spatial_summary)


Loaded Census ZCTA Reference: 404 NYC ZIP codes mapped to centroids.


,zip5,centroid_lat,centroid_lon,grid_lat,grid_lon,grid_id,source
0,10001,40.751,-73.997,40.800,-74.000,GRID_40.8_-74.0,US_CENSUS_BUREAU_GAZETTEER_2020
1,10002,40.716,-73.987,40.700,-74.000,GRID_40.7_-74.0,US_CENSUS_BUREAU_GAZETTEER_2020
2,10003,40.732,-73.989,40.700,-74.000,GRID_40.7_-74.0,US_CENSUS_BUREAU_GAZETTEER_2020
3,10004,40.690,-74.017,40.700,-74.000,GRID_40.7_-74.0,US_CENSUS_BUREAU_GAZETTEER_2020



--- Silver 311 Spatial Resolution Breakdown ---


,Record Count,Percentage (%)
spatial_resolution_level,,
EXACT_COORDINATES,89665,98.350
ZIP_APPROXIMATION,990,1.090
BOROUGH_APPROXIMATION,423,0.460
UNLOCATED,89,0.100


In [11]:
# DECISION 8: Quarantine Segregation (Temporal Validation Failures)
# Records that fail core logical invariants (e.g. closed before created) are quarantined to data/silver/quarantine/

quarantine_files = sorted(QUARANTINE_DIR.glob("**/*.parquet"))
print(f"Found {len(quarantine_files)} Quarantine Parquet file(s):")

if quarantine_files:
    df_quarantine = pd.concat([pd.read_parquet(f) for f in quarantine_files], ignore_index=True)
    print(f"Total Quarantined Records: {len(df_quarantine):,}")
    print("\n--- Quarantined Records Breakdown by Failure Reason ---")
    display(df_quarantine["quarantine_reason"].value_counts())
    
    print("\n--- Sample: Quarantined Records with Negative Resolution Duration ---")
    q_cols = ["unique_key", "created_date", "closed_date", "resolution_duration_hours", "quarantine_reason"]
    display(df_quarantine[q_cols].head(5))
else:
    print("Quarantine directory is clean (0 invalid records).")


Found 1 Quarantine Parquet file(s):
Total Quarantined Records: 50

--- Quarantined Records Breakdown by Failure Reason ---


quarantine_reason
NEGATIVE_RESOLUTION_DURATION    50
Name: count, dtype: int64


--- Sample: Quarantined Records with Negative Resolution Duration ---


,unique_key,created_date,closed_date,resolution_duration_hours,quarantine_reason
0,70764109,2026-10-15T08:01:21.000,2026-10-15T08:01:00.000,-0.010,NEGATIVE_RESOLUTION_DURATION
1,70762670,2026-10-15T08:06:20.000,2026-10-15T08:06:00.000,-0.010,NEGATIVE_RESOLUTION_DURATION
2,70771343,2026-10-15T08:59:34.000,2026-10-15T08:59:00.000,-0.010,NEGATIVE_RESOLUTION_DURATION
3,70772807,2026-10-15T09:34:33.000,2026-10-15T09:34:00.000,-0.010,NEGATIVE_RESOLUTION_DURATION
4,70772804,2026-10-15T13:01:20.000,2026-10-15T13:01:00.000,-0.010,NEGATIVE_RESOLUTION_DURATION


## Part 3: Inspecting the Whole SILVER Layer
The Silver layer provides validated, enriched, and standardized analytical tables ready for querying, reporting, and feature engineering.


In [12]:
# TABLE 3A: Clean NYC 311 Silver Table (data/silver/nyc311/)
print(f"Total Clean 311 Records in Silver: {len(df_silver_311):,}")
print(f"Total Clean 311 Columns in Silver: {len(df_silver_311.columns)}")

silver_311_cols = [
    "unique_key", "created_at_utc", "closed_at_utc", "is_ticket_closed", 
    "ticket_status_category", "resolution_duration_hours", "open_ticket_age_days",
    "agency", "complaint_type", "borough", "clean_latitude", "clean_longitude", 
    "grid_id", "spatial_resolution_level", "weather_join_hour_utc"
]
print("\n--- Clean NYC 311 Silver Table (Top 10 Records) ---")
display(df_silver_311[silver_311_cols].head(10))

print("\n--- Resolution Duration Distribution (Closed Tickets, Hours) ---")
display(df_silver_311[df_silver_311["is_ticket_closed"]]["resolution_duration_hours"].describe())


Total Clean 311 Records in Silver: 91,167
Total Clean 311 Columns in Silver: 68

--- Clean NYC 311 Silver Table (Top 10 Records) ---


,unique_key,created_at_utc,closed_at_utc,is_ticket_closed,ticket_status_category,resolution_duration_hours,open_ticket_age_days,agency,complaint_type,borough,clean_latitude,clean_longitude,grid_id,spatial_resolution_level,weather_join_hour_utc
0,69734449,2026-07-15 22:35:39+00:00,2026-07-16 01:40:32+00:00,True,CLOSED,3.080,NaN,NYPD,Noise - Residential,BRONX,40.834,-73.905,GRID_40.8_-73.9,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
1,69734931,2026-07-15 22:35:48+00:00,2026-07-16 21:52:28+00:00,True,CLOSED,23.280,NaN,DPR,Damaged Tree,BROOKLYN,40.627,-74.034,GRID_40.6_-74.0,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
2,69726765,2026-07-15 22:36:00+00:00,2026-07-15 22:36:00+00:00,True,CLOSED,0.000,NaN,DEP,Water System,MANHATTAN,40.802,-73.957,GRID_40.8_-74.0,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
3,69727719,2026-07-15 22:36:39+00:00,NaT,False,OPEN,NaN,101.830,DPR,Damaged Tree,BROOKLYN,40.604,-74.013,GRID_40.6_-74.0,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
4,69729114,2026-07-15 22:36:40+00:00,2026-07-16 01:12:48+00:00,True,CLOSED,2.600,NaN,NYPD,Abandoned Vehicle,BROOKLYN,40.638,-73.914,GRID_40.6_-73.9,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
5,69730122,2026-07-15 22:36:49+00:00,2026-07-15 23:46:51+00:00,True,CLOSED,1.170,NaN,NYPD,Noise - Residential,BROOKLYN,40.677,-73.935,GRID_40.7_-73.9,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
6,69727731,2026-07-15 22:36:53+00:00,NaT,False,OPEN,NaN,101.830,DPR,Dead/Dying Tree,QUEENS,40.748,-73.779,GRID_40.7_-73.8,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
7,69732781,2026-07-15 22:37:03+00:00,2026-07-16 01:59:52+00:00,True,CLOSED,3.380,NaN,NYPD,Illegal Parking,QUEENS,40.769,-73.875,GRID_40.8_-73.9,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
8,69731291,2026-07-15 22:37:21+00:00,2026-07-16 01:21:41+00:00,True,CLOSED,2.740,NaN,NYPD,Illegal Parking,BROOKLYN,40.585,-73.936,GRID_40.6_-73.9,EXACT_COORDINATES,2026-07-15 22:00:00+00:00
9,69730105,2026-07-15 22:37:24+00:00,2026-07-15 22:51:20+00:00,True,CLOSED,0.230,NaN,NYPD,Noise - Residential,BROOKLYN,40.608,-73.999,GRID_40.6_-74.0,EXACT_COORDINATES,2026-07-15 22:00:00+00:00



--- Resolution Duration Distribution (Closed Tickets, Hours) ---


count   64661.000
mean       37.690
std       149.307
min         0.000
25%         0.760
50%         2.820
75%        20.420
max      2136.030
Name: resolution_duration_hours, dtype: float64

In [13]:
# TABLE 3B: Clean Weather Silver Table (data/silver/weather/)
silver_weather_files = sorted(SILVER_DIR.glob("weather/**/*.parquet"))
df_silver_weather = pd.concat([pd.read_parquet(f) for f in silver_weather_files], ignore_index=True)

print(f"Total Clean Weather Observations in Silver: {len(df_silver_weather):,}")
print(f"Total Clean Weather Columns in Silver: {len(df_silver_weather.columns)}")

silver_w_cols = [
    "grid_id", "timestamp_utc", "temperature_2m", "apparent_temperature", 
    "precipitation", "precip_past_3h_mm", "precip_past_24h_mm", 
    "wind_speed_10m", "max_wind_gust_past_6h_ms", "min_temp_past_24h_c", 
    "is_freezing", "is_extreme_weather"
]
print("\n--- Clean Weather Silver Table (Top 10 Observations) ---")
display(df_silver_weather[silver_w_cols].head(10))

print("\n--- Weather Metric Distributions (Metric Units: °C and m/s) ---")
display(df_silver_weather[["temperature_2m", "precipitation", "precip_past_3h_mm", "wind_speed_10m", "max_wind_gust_past_6h_ms"]].describe())


Total Clean Weather Observations in Silver: 4,560
Total Clean Weather Columns in Silver: 38

--- Clean Weather Silver Table (Top 10 Observations) ---


,grid_id,timestamp_utc,temperature_2m,apparent_temperature,precipitation,precip_past_3h_mm,precip_past_24h_mm,wind_speed_10m,max_wind_gust_past_6h_ms,min_temp_past_24h_c,is_freezing,is_extreme_weather
0,GRID_40.5_-74.1,2026-07-15 00:00:00+00:00,31.000,32.100,0.000,0.000,0.000,20.600,49.300,31.000,False,True
1,GRID_40.5_-74.1,2026-07-15 01:00:00+00:00,30.200,31.200,0.000,0.000,0.000,21.000,49.300,30.200,False,True
2,GRID_40.5_-74.1,2026-07-15 02:00:00+00:00,29.400,30.500,0.000,0.000,0.000,21.100,49.300,29.400,False,True
3,GRID_40.5_-74.1,2026-07-15 03:00:00+00:00,28.700,30.100,0.000,0.000,0.000,19.500,49.300,28.700,False,True
4,GRID_40.5_-74.1,2026-07-15 04:00:00+00:00,28.000,29.500,0.000,0.000,0.000,18.000,49.300,28.000,False,True
5,GRID_40.5_-74.1,2026-07-15 05:00:00+00:00,27.400,28.900,0.000,0.000,0.000,17.400,49.300,27.400,False,True
6,GRID_40.5_-74.1,2026-07-15 06:00:00+00:00,27.000,29.100,0.000,0.000,0.000,15.800,42.800,27.000,False,True
7,GRID_40.5_-74.1,2026-07-15 07:00:00+00:00,26.700,28.900,0.000,0.000,0.000,15.300,42.800,26.700,False,True
8,GRID_40.5_-74.1,2026-07-15 08:00:00+00:00,26.200,28.500,0.000,0.000,0.000,13.900,42.100,26.200,False,True
9,GRID_40.5_-74.1,2026-07-15 09:00:00+00:00,25.600,27.900,0.000,0.000,0.000,13.700,38.900,25.600,False,True



--- Weather Metric Distributions (Metric Units: °C and m/s) ---


,temperature_2m,precipitation,precip_past_3h_mm,wind_speed_10m,max_wind_gust_past_6h_ms
count,4560.000,4560.000,4560.000,4560.000,4560.000
mean,18.501,0.055,0.164,11.083,31.778
std,7.931,0.483,0.925,4.663,10.385
min,3.000,0.000,0.000,0.200,6.800
25%,12.300,0.000,0.000,7.300,24.100
50%,17.300,0.000,0.000,11.400,33.100
75%,24.700,0.000,0.000,14.800,40.000
max,37.600,15.700,16.800,23.100,53.300


In [14]:
# TABLE 3C: Unified Analytical Fact Table (data/silver/311_weather_joined/)
fact_files = sorted(SILVER_DIR.glob("311_weather_joined/**/*.parquet"))
df_fact = pd.concat([pd.read_parquet(f) for f in fact_files], ignore_index=True)

print(f"Total Fact Table Records: {len(df_fact):,}")
print(f"Total Fact Table Columns: {len(df_fact.columns)}")

matched_count = int(df_fact["is_weather_matched"].sum())
total_count = len(df_fact)
match_rate = (matched_count / total_count) * 100.0

print(f"\n--- Fact Table Weather Match Summary ---")
print(f"Enriched with Local Weather : {matched_count:,} ({match_rate:.2f}%)")
print(f"Unmatched (Historical/No Weather) : {total_count - matched_count:,} ({100 - match_rate:.2f}%)")

fact_sample_cols = [
    "unique_key", "created_at_utc", "complaint_type", "borough", "grid_id", 
    "is_ticket_closed", "resolution_duration_hours", 
    "temperature_2m", "precipitation", "precip_past_3h_mm", 
    "max_wind_gust_past_6h_ms", "is_extreme_weather"
]
print("\n--- Unified Analytical Fact Table (Top 10 Enriched Records) ---")
display(df_fact[fact_sample_cols].head(10))

# Sample Insight: Weather-Sensitive Complaints vs. Rain/Wind
top_complaints = ["Noise", "Illegal Parking", "Heat/Hot Water", "Blocked Driveway", "Street Condition", "Sewer", "Water System"]
weather_by_type = df_fact[df_fact["complaint_type"].isin(top_complaints)].groupby("complaint_type")[["temperature_2m", "precipitation", "precip_past_3h_mm", "max_wind_gust_past_6h_ms"]].mean()

print("\n--- Average Weather Metrics by Top Complaint Category ---")
display(weather_by_type.round(2))


Total Fact Table Records: 91,167
Total Fact Table Columns: 91

--- Fact Table Weather Match Summary ---
Enriched with Local Weather : 78,726 (86.35%)
Unmatched (Historical/No Weather) : 12,441 (13.65%)

--- Unified Analytical Fact Table (Top 10 Enriched Records) ---


,unique_key,created_at_utc,complaint_type,borough,grid_id,is_ticket_closed,resolution_duration_hours,temperature_2m,precipitation,precip_past_3h_mm,max_wind_gust_past_6h_ms,is_extreme_weather
0,69734449,2026-07-15 22:35:39+00:00,Noise - Residential,BRONX,GRID_40.8_-73.9,True,3.080,32.900,0.000,0.200,48.200,True
1,69734931,2026-07-15 22:35:48+00:00,Damaged Tree,BROOKLYN,GRID_40.6_-74.0,True,23.280,33.500,0.400,0.500,52.900,True
2,69726765,2026-07-15 22:36:00+00:00,Water System,MANHATTAN,GRID_40.8_-74.0,True,0.000,32.200,0.000,0.900,46.800,True
3,69727719,2026-07-15 22:36:39+00:00,Damaged Tree,BROOKLYN,GRID_40.6_-74.0,False,NaN,33.500,0.400,0.500,52.900,True
4,69729114,2026-07-15 22:36:40+00:00,Abandoned Vehicle,BROOKLYN,GRID_40.6_-73.9,True,2.600,33.500,0.400,0.500,52.900,True
5,69730122,2026-07-15 22:36:49+00:00,Noise - Residential,BROOKLYN,GRID_40.7_-73.9,True,1.170,33.800,0.000,0.300,53.300,True
6,69727731,2026-07-15 22:36:53+00:00,Dead/Dying Tree,QUEENS,GRID_40.7_-73.8,False,NaN,34.300,0.200,0.400,52.600,True
7,69732781,2026-07-15 22:37:03+00:00,Illegal Parking,QUEENS,GRID_40.8_-73.9,True,3.380,32.900,0.000,0.200,48.200,True
8,69731291,2026-07-15 22:37:21+00:00,Illegal Parking,BROOKLYN,GRID_40.6_-73.9,True,2.740,33.500,0.400,0.500,52.900,True
9,69730105,2026-07-15 22:37:24+00:00,Noise - Residential,BROOKLYN,GRID_40.6_-74.0,True,0.230,33.500,0.400,0.500,52.900,True



--- Average Weather Metrics by Top Complaint Category ---


,temperature_2m,precipitation,precip_past_3h_mm,max_wind_gust_past_6h_ms
complaint_type,,,,
Blocked Driveway,15.090,0.090,0.250,30.970
Heat/Hot Water,10.910,0.020,0.120,26.640
Illegal Parking,15.120,0.060,0.210,30.240
Noise,14.130,0.070,0.150,27.890
Sewer,28.880,0.030,0.110,35.050
Street Condition,16.950,0.080,0.240,31.160
Water System,28.470,0.020,0.110,35.610


## Part 4: Technical Comparison Matrix (Bronze vs. Silver)
The table below summarizes the transformation matrix across both datasets from raw ingestion to clean analytical presentation.


In [15]:
comparison_matrix = pd.DataFrame([
    {
        "Dimension": "Storage Format & Compression",
        "Bronze NYC 311": "Snappy Parquet (Partitioned YYYY/MM)",
        "Silver NYC 311": "Snappy Parquet (Partitioned YYYY/MM)",
        "Bronze Weather": "Snappy Parquet (grid_id/YYYY)",
        "Silver Weather": "Snappy Parquet (grid_id/YYYY)",
    },
    {
        "Dimension": "File Sizing & Layout",
        "Bronze NYC 311": "Compacted coalesced partitions",
        "Silver NYC 311": "Optimized single partition files",
        "Bronze Weather": "Compacted coalesced partitions",
        "Silver Weather": "1 file per grid cell / year",
    },
    {
        "Dimension": "Temporal Semantics",
        "Bronze NYC 311": "Raw floating string (America/New_York)",
        "Silver NYC 311": "Standardized UTC timestamp + local hour",
        "Bronze Weather": "UTC ISO string",
        "Silver Weather": "Standardized UTC timestamp",
    },
    {
        "Dimension": "Spatial Resolution",
        "Bronze NYC 311": "Raw strings (may be null/0.0/-999)",
        "Silver NYC 311": "3-Level Hierarchical Imputation",
        "Bronze Weather": "0.1° (~11 km) native ERA5 grid",
        "Silver Weather": "19 NYC spatial grid centroids",
    },
    {
        "Dimension": "Quality & Validation",
        "Bronze NYC 311": "Preserved raw as-is (immutable)",
        "Silver NYC 311": "Negative duration quarantined",
        "Bronze Weather": "Raw sensor readings",
        "Silver Weather": "Column nullification + Extreme flags",
    },
    {
        "Dimension": "Derived Engineering Features",
        "Bronze NYC 311": "Tier 2 Audit Metadata (_raw_row_hash)",
        "Silver NYC 311": "is_ticket_closed, open_ticket_age_days",
        "Bronze Weather": "Tier 2 Audit Metadata (_raw_row_hash)",
        "Silver Weather": "precip_past_3h, 6h, 24h, max_gust_6h",
    },
    {
        "Dimension": "Analytical Usability",
        "Bronze NYC 311": "System audit and raw recovery",
        "Silver NYC 311": "Clean dimension ready for analytics",
        "Bronze Weather": "System audit and raw recovery",
        "Silver Weather": "Broadcast dimension for fact table join",
    },
])

print("==========================================================================================")
print("DAT535 MEDALLION PIPELINE: COMPREHENSIVE BRONZE VS. SILVER ARCHITECTURAL MATRIX")
print("==========================================================================================")
display(comparison_matrix.set_index("Dimension"))


DAT535 MEDALLION PIPELINE: COMPREHENSIVE BRONZE VS. SILVER ARCHITECTURAL MATRIX


,Bronze NYC 311,Silver NYC 311,Bronze Weather,Silver Weather
Dimension,,,,
Storage Format & Compression,Snappy Parquet (Partitioned YYYY/MM),Snappy Parquet (Partitioned YYYY/MM),Snappy Parquet (grid_id/YYYY),Snappy Parquet (grid_id/YYYY)
File Sizing & Layout,Compacted coalesced partitions,Optimized single partition files,Compacted coalesced partitions,1 file per grid cell / year
Temporal Semantics,Raw floating string (America/New_York),Standardized UTC timestamp + local hour,UTC ISO string,Standardized UTC timestamp
Spatial Resolution,Raw strings (may be null/0.0/-999),3-Level Hierarchical Imputation,0.1° (~11 km) native ERA5 grid,19 NYC spatial grid centroids
Quality & Validation,Preserved raw as-is (immutable),Negative duration quarantined,Raw sensor readings,Column nullification + Extreme flags
Derived Engineering Features,Tier 2 Audit Metadata (_raw_row_hash),"is_ticket_closed, open_ticket_age_days",Tier 2 Audit Metadata (_raw_row_hash),"precip_past_3h, 6h, 24h, max_gust_6h"
Analytical Usability,System audit and raw recovery,Clean dimension ready for analytics,System audit and raw recovery,Broadcast dimension for fact table join
